# AC-Zero — Dataset Generation (Kaggle)

Grows a persistent, guaranteed-solvable Andrews-Curtis dataset outward from the
trivial group with `aczero dataset grow`, until the Kaggle time budget is nearly
spent, then writes the dataset **and** a statistics summary to `/kaggle/working`
(the notebook's saved output).

**Before you run**
- Settings -> **Internet: On** (needed to `pip install` from GitHub).
- Accelerator: **None / CPU** is enough — generation is CPU graph search, no GPU.
- Kaggle sessions are capped at ~12 h. `TIME_BUDGET_HOURS` below stops the grow
  with a safety margin so the dataset is always flushed to disk before the kill.
- Grow is **resumable**: to continue a previous run beyond one session, attach
  that run's output dataset as an input and point `RESUME_FROM` at its `.json`.

## 1. Configuration

In [ ]:
import os

# --- Repository -------------------------------------------------------------
REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"  # branch / tag / commit to install

# --- Dataset shape ----------------------------------------------------------
RANK = 2  # group rank (catalog has 3*rank^2 primitive moves)
TOTAL_LENGTH_CAP = 48  # discard presentations longer than this
SELECT = "smallest"  # frontier order: "smallest" | "weighted-random"
SEED = 0
WORKERS = 0  # 0 = all CPU cores; 1 = single in-process worker

# --- Time budget (Kaggle hard-kills at ~12 h) -------------------------------
TIME_BUDGET_HOURS = 11.5  # stop growing after this much wall-clock
SAFETY_MARGIN_MIN = 10  # head-room reserved for the final flush + summary

# --- Growth loop ------------------------------------------------------------
TARGET_PER_CHUNK = 2000  # new groups added per grow() call
CHECKPOINT_EVERY = 1000  # grow() flushes to disk every N new groups

# --- Output / resume --------------------------------------------------------
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATASET_NAME = f"train_rank{RANK}.json"
RESUME_FROM = ""  # e.g. "/kaggle/input/ac-zero-dataset/train_rank2.json"

# --- Optional difficulty pass (needs `dataset descent`, not on `main`) ------
RUN_DESCENT = False

## 2. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys

spec = f"git+{REPO_URL}@{REPO_BRANCH}"
# --no-deps keeps Kaggle's pre-built torch/numpy; we add only the light deps the
# generation path actually needs (the grow search itself is pure-Python).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", spec], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy>=1.26",
        "pydantic>=2",
        "pyyaml>=6",
        "typer>=0.12",
        "rich>=13",
        "gymnasium>=0.29",
        "matplotlib>=3.8",
    ],
    check=True,
)

print("Installed ac_zero from", spec)

## 3. Time budget

In [ ]:
import time

START_TIME = time.monotonic()
DEADLINE = START_TIME + TIME_BUDGET_HOURS * 3600
MARGIN_S = SAFETY_MARGIN_MIN * 60


def elapsed_s():
    return time.monotonic() - START_TIME


def time_left_s():
    return DEADLINE - time.monotonic()


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


class DeadlineReached(Exception):
    """Raised from a progress callback to stop cleanly before the Kaggle kill."""


print(
    f"Budget {TIME_BUDGET_HOURS} h with a {SAFETY_MARGIN_MIN} min safety margin "
    f"(stop growing when {hms(MARGIN_S)} remain)."
)

## 4. Grow the dataset

In [ ]:
import shutil
from pathlib import Path

from ac_zero.datasets.grow import GrowConfig, grow_dataset

out_path = Path(OUTPUT_DIR) / DATASET_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)

# Optional resume: seed the working file from a previous session's output; grow
# then continues from the accumulated frontier and only ever adds groups.
if RESUME_FROM and Path(RESUME_FROM).exists() and Path(RESUME_FROM).resolve() != out_path.resolve():
    shutil.copy(RESUME_FROM, out_path)
    print("Resuming from", RESUME_FROM)


def progress(message, metrics):
    # grow() checkpoints between rounds, so raising here loses at most the groups
    # added since the last checkpoint — the dataset on disk stays consistent.
    if time_left_s() <= MARGIN_S:
        raise DeadlineReached(message)
    if message in ("checkpoint", "grow complete") or "added" in metrics:
        print(f"[{hms(elapsed_s())}] {message}: {metrics}")


total_added = 0
chunks = 0
try:
    while time_left_s() > MARGIN_S:
        cfg = GrowConfig(
            rank=RANK,
            target=TARGET_PER_CHUNK,
            select=SELECT,
            seed=SEED,
            total_length_cap=TOTAL_LENGTH_CAP,
            workers=WORKERS,
            checkpoint_every=CHECKPOINT_EVERY,
            log_every=max(1, TARGET_PER_CHUNK // 4),
        )
        report = grow_dataset(out_path, cfg, progress=progress)
        total_added += report.added
        chunks += 1
        print(
            f"[{hms(elapsed_s())}] chunk {chunks}: +{report.added} groups "
            f"(total {report.total}, frontier {report.frontier}, max depth {report.max_difficulty})"
        )
        if report.added == 0:  # frontier exhausted within the length cap
            print("Frontier exhausted within the length cap — nothing left to add.")
            break
except DeadlineReached as stop:
    print(f"[{hms(elapsed_s())}] time budget reached ({stop}); dataset flushed to disk.")

print(f"\nAdded ~{total_added} groups across {chunks} chunk(s) in {hms(elapsed_s())}.")
print("Dataset:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")

## 5. Optional — descent-difficulty annotation

Only available when `REPO_BRANCH` ships `aczero dataset descent` (not on `main`).
Set `RUN_DESCENT = True` in the config cell to annotate each entry with its
length-descent distance; the pass is resumable and time-boxed like the grow.

In [ ]:
if not RUN_DESCENT:
    print("Skipping descent annotation (RUN_DESCENT = False).")
else:
    try:
        from ac_zero.datasets.descent import DescentAnnotateConfig, annotate_descent
    except Exception as exc:
        print("`dataset descent` is not available on this ref:", exc)
    else:
        dcfg = DescentAnnotateConfig(
            total_length_cap=TOTAL_LENGTH_CAP,
            max_depth=32,
            max_expansions=20_000,
            workers=WORKERS,
            checkpoint_every=CHECKPOINT_EVERY,
        )

        def dprogress(message, metrics):
            if time_left_s() <= MARGIN_S:
                raise DeadlineReached(message)
            print(f"[{hms(elapsed_s())}] {message}: {metrics}")

        try:
            drep = annotate_descent(out_path, dcfg, output=out_path, progress=dprogress)
            print("descent:", drep)
        except DeadlineReached:
            print("descent annotation stopped at the time budget (partial, checkpointed).")

## 6. Dataset summary & output artifacts

In [ ]:
import json
from collections import Counter

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

data = json.loads(out_path.read_text())
instances = data.get("instances", [])


def dist(values):
    counts = Counter(values)
    pop = sum(counts.values())
    mean = (sum(v * n for v, n in counts.items()) / pop) if pop else None
    return {
        "counts": dict(sorted(counts.items())),
        "population": pop,
        "min": min(counts) if counts else None,
        "max": max(counts) if counts else None,
        "mean": mean,
    }


difficulty = [int(e.get("difficulty", 0)) for e in instances]
lengths = [sum(len(r) for r in e.get("relators", [])) for e in instances]
predec = [len(e.get("predecessors", [])) for e in instances]
known_ops = [
    int(e["minimal_known_operations"])
    for e in instances
    if e.get("minimal_known_operations") is not None
]
exhausted = sum(1 for e in instances if e.get("exhausted"))

stats = {
    "rank": data.get("rank"),
    "schema_version": data.get("schema_version"),
    "generator": (data.get("provenance") or {}).get("generator"),
    "total_groups": len(instances),
    "roots_depth0": sum(1 for d in difficulty if d == 0),
    "frontier_open": len(instances) - exhausted,
    "exhausted": exhausted,
    "with_known_trivialization": len(known_ops),
    "generated_seconds": round(elapsed_s(), 1),
    "difficulty": dist(difficulty),
    "total_length": dist(lengths),
    "co_optimal_predecessors": dist(predec),
    "known_operations": dist(known_ops),
}
(Path(OUTPUT_DIR) / "dataset_stats.json").write_text(json.dumps(stats, indent=2))


def table(title, unit, d):
    lines = [f"### {title}", ""]
    if not d["population"]:
        return lines + ["_No data._", ""]
    lines.append(f"- min {d['min']} · max {d['max']} · mean {d['mean']:.2f}")
    lines += ["", f"| {unit} | groups |", "| --- | --- |"]
    lines += [f"| {k} | {v} |" for k, v in d["counts"].items()]
    return lines + [""]


report_md = [
    f"# Dataset summary: {DATASET_NAME}",
    "",
    f"- Generator: `{stats['generator']}`",
    f"- Schema: `{stats['schema_version']}`",
    f"- Rank: {stats['rank']}",
    f"- Total groups: {stats['total_groups']}",
    f"- Roots (depth 0): {stats['roots_depth0']}",
    f"- Frontier open: {stats['frontier_open']} · Exhausted: {stats['exhausted']}",
    f"- With known trivialization: {stats['with_known_trivialization']}",
    f"- Wall-clock: {hms(elapsed_s())}",
    "",
]
report_md += table("By construction difficulty (depth from trivial)", "depth", stats["difficulty"])
report_md += table("By size (total relator length)", "length", stats["total_length"])
report_md += table("By co-optimal construction moves", "moves", stats["co_optimal_predecessors"])
report_md += table("By known trivialization length", "operations", stats["known_operations"])
(Path(OUTPUT_DIR) / "dataset_summary.md").write_text("\n".join(report_md) + "\n")

for key, label, fname in [
    ("difficulty", "construction depth (from trivial)", "hist_difficulty.png"),
    ("total_length", "total relator length", "hist_length.png"),
]:
    d = stats[key]
    if d["population"]:
        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.bar(list(d["counts"].keys()), list(d["counts"].values()))
        ax.set_xlabel(label)
        ax.set_ylabel("groups")
        ax.set_title(f"Groups by {label}")
        fig.tight_layout()
        fig.savefig(Path(OUTPUT_DIR) / fname, dpi=120)
        plt.close(fig)

display(Markdown("\n".join(report_md)))
print("\nOutput artifacts in", OUTPUT_DIR, ":")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    if p.is_file():
        print(f"  {p.name:28s} {p.stat().st_size / 1e3:9.1f} KB")